In [1]:
import duckdb

In [2]:
duckdb.read_csv("../raw_data/google_ads/*.csv")

┌─────────────┬─────────────┬──────────────────────┬─────────────┬───────────┬─────────────┬────────┬──────────────┐
│    date     │ campaign_id │    campaign_name     │   channel   │ spend_usd │ impressions │ clicks │ soft_deleted │
│   varchar   │    int64    │       varchar        │   varchar   │  double   │    int64    │ int64  │   boolean    │
├─────────────┼─────────────┼──────────────────────┼─────────────┼───────────┼─────────────┼────────┼──────────────┤
│ 04-Apr-2025 │           1 │ Brand Awareness Q1   │ Search      │    281.27 │       31784 │    814 │ false        │
│ 2025-04-04  │           5 │ New User Acquisition │ Search      │     299.1 │       31775 │   1441 │ false        │
│ 2025-04-04  │          21 │ Dynamic Search Ads   │ Paid Search │    238.37 │       43935 │   2773 │ false        │
│ 04/05/2025  │           1 │ Brand Awareness Q1   │ SEARCH      │    317.87 │       39921 │   1402 │ false        │
│ 05-Apr-2025 │           5 │ New User Acquisition │ Paid Search

In [3]:
conn = duckdb.connect("marketing_data.duckdb")

conn.execute(""" CREATE SCHEMA IF NOT EXISTS raw""")

conn.execute("""
    CREATE TABLE IF NOT EXISTS raw.google_ads AS
    SELECT * FROM read_csv('../raw_data/google_ads/*.csv')
""")

In [4]:
conn.execute("show all tables").fetchdf()

,database,schema,name,column_names,column_types,temporary
0,marketing_data,raw,app_conversions,"[conversion_id, usr_id, cmpgn_id, conv_type_cd...","[INTEGER, INTEGER, INTEGER, INTEGER, DECIMAL(1...",False
1,marketing_data,raw,google_ads,"[date, campaign_id, campaign_name, channel, sp...","[VARCHAR, BIGINT, VARCHAR, VARCHAR, DOUBLE, BI...",False
2,marketing_data,raw,segment_tracks,"[message_id, type, event, timestamp, user_id, ...","[UUID, VARCHAR, VARCHAR, VARCHAR, VARCHAR, VAR...",False


In [5]:
conn.execute("select * from raw.google_ads limit 10").fetchdf()

,date,campaign_id,campaign_name,channel,spend_usd,impressions,clicks,soft_deleted
0,04-Apr-2025,1,Brand Awareness Q1,Search,281.27,31784,814,False
1,2025-04-04,5,New User Acquisition,Search,299.10,31775,1441,False
2,2025-04-04,21,Dynamic Search Ads,Paid Search,238.37,43935,2773,False
3,04/05/2025,1,Brand Awareness Q1,SEARCH,317.87,39921,1402,False
4,05-Apr-2025,5,New User Acquisition,Paid Search,509.65,74533,5072,False
5,2025-04-05,21,Dynamic Search Ads,SEARCH,167.15,19176,298,False
6,06-Apr-2025,1,Brand Awareness Q1,Paid Search,287.15,30191,866,False
7,06-Apr-2025,5,New User Acquisition,search,335.86,37748,2822,False
8,06-Apr-2025,21,Dynamic Search Ads,SEARCH,173.32,19434,481,False
9,07-Apr-2025,1,Brand Awareness Q1,SEARCH,421.91,47163,1285,False


In [6]:
conn.execute("select count(*) from raw.google_ads").fetchdf()

,count_star()
0,2088


In [7]:
conn.execute("select campaign_name, count(*) from raw.google_ads group by campaign_name").fetchdf()

,campaign_name,count_star()
0,Brand Awareness Q1,17
1,Dynamic Search Ads,46
2,New User Acquisition,4
3,App Install Campaign,215
4,Back to School,6
5,Holiday Promo,51
6,New User Acquisition,310
7,Brand Awareness Q1,71
8,Product Launch - Video,84
9,Competitor Conquest,288


In [8]:
conn.execute("select distinct channel from raw.google_ads").fetchdf()

,channel
0,Paid Search
1,product_shopping
2,search
3,video
4,Display
5,SHOPPING
6,Search
7,youtube_video
8,display_network
9,shopping


In [9]:
duckdb.read_json("../raw_data/segment/segment_tracks.jsonl")

┌──────────────────────────────────┬─────────┬────────────────┬─────────────────────────┬─────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────┐
│            message_id            │  type   │     event      │        timestamp        │ user_id │ anonymous_id │                                                                                                                                   properties                                                                                                                                   │                                       context                                       │
│             varchar              │ varchar │    va

In [10]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS raw.segment_tracks AS
    SELECT 
        message_id,
        type,
        event,
        timestamp,
        user_id,
        anonymous_id,
        json(properties) as properties,
        json(context) as context,
        session_id
             
    FROM read_json('../raw_data/segment/segment_tracks.jsonl', sample_size=-1)

""")

**sample_size=-1 to force DuckDB to scan the entire file before inferring the schema**



[DuckDB Loading JSON Documentation](https://duckdb.org/docs/current/data/json/loading_json)

In [11]:
conn.execute("describe raw.segment_tracks").fetchdf()

,column_name,column_type,null,key,default,extra
0,message_id,UUID,YES,None,None,None
1,type,VARCHAR,YES,None,None,None
2,event,VARCHAR,YES,None,None,None
3,timestamp,VARCHAR,YES,None,None,None
4,user_id,VARCHAR,YES,None,None,None
5,anonymous_id,VARCHAR,YES,None,None,None
6,properties,JSON,YES,None,None,None
7,context,JSON,YES,None,None,None
8,session_id,VARCHAR,YES,None,None,None


In [12]:
conn.execute("select * from raw.segment_tracks where message_id = 'e12db95e35b3a3ba7781549134c83a37' ").fetch_df()

,message_id,type,event,timestamp,user_id,anonymous_id,properties,context,session_id
0,e12db95e-35b3-a3ba-7781-549134c83a37,track,product_view,2025-04-04 11:03:56 UTC,u_04441,anon_292484,"{""campaign_id"":""8"",""channel"":""display"",""page_u...","{""library_name"":""analytics.js"",""library_versio...",None


In [13]:

conn.execute("select * from raw.segment_tracks where message_id = 'fde6a9fa2641d21500f4862e5f687ec4' ").fetch_df()

,message_id,type,event,timestamp,user_id,anonymous_id,properties,context,session_id
0,fde6a9fa-2641-d215-00f4-862e5f687ec4,track,page_view,2025-07-03 14:22:06 UTC,u_04671,anon_698079,"{""campaign_id"":""5"",""channel"":""search"",""page_ur...","{""library_name"":""analytics.js"",""library_versio...",sess_84599834


In [14]:
conn.execute("INSTALL postgres; LOAD postgres;")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [15]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS raw.app_conversions AS 
    SELECT * FROM postgres_scan(
        'host=localhost port=5432 dbname=marketing_attribution user=analytics password=analytics',
        'public',
        'app_conversions'
    )
""")

In [16]:
conn.execute("show all tables").fetch_df()

,database,schema,name,column_names,column_types,temporary
0,marketing_data,raw,app_conversions,"[conversion_id, usr_id, cmpgn_id, conv_type_cd...","[INTEGER, INTEGER, INTEGER, INTEGER, DECIMAL(1...",False
1,marketing_data,raw,google_ads,"[date, campaign_id, campaign_name, channel, sp...","[VARCHAR, BIGINT, VARCHAR, VARCHAR, DOUBLE, BI...",False
2,marketing_data,raw,segment_tracks,"[message_id, type, event, timestamp, user_id, ...","[UUID, VARCHAR, VARCHAR, VARCHAR, VARCHAR, VAR...",False


In [17]:
conn.execute("select * from raw.app_conversions limit 10").fetch_df()

,conversion_id,usr_id,cmpgn_id,conv_type_cd,revenue_amt,conv_ts,created_at
0,1,4197,1,1,365.30,2025-04-04 18:00:32+00:00,2025-04-04T18:00:32Z
1,2,2512,20,2,NaN,2025-04-04 17:51:03+00:00,2025-04-04T17:51:03Z
2,3,1178,21,3,NaN,2025-04-04 15:26:21+00:00,2025-04-04T15:26:21Z
3,4,3425,1,3,NaN,2025-04-04 16:27:09+00:00,2025-04-04T16:27:09Z
4,5,1733,21,3,NaN,2025-04-04 09:14:53+00:00,2025-04-04T09:14:53Z
5,6,2077,11,1,269.98,2025-04-04 16:18:23+00:00,2025-04-04T16:18:23Z
6,7,4859,1,4,3635.38,2025-04-04 02:40:27+00:00,2025-04-04T02:40:27Z
7,8,2702,21,4,4936.31,2025-04-04 07:16:45+00:00,2025-04-04T07:16:45Z
8,9,1355,15,2,NaN,2025-04-04 15:01:13+00:00,2025-04-04T15:01:13Z
9,10,4980,11,1,10.77,2025-04-04 09:56:43+00:00,2025-04-04T09:56:43Z


In [18]:
conn.sql("call start_ui()")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┐
│                result                │
│               varchar                │
├──────────────────────────────────────┤
│ UI started at http://localhost:4213/ │
└──────────────────────────────────────┘

In [19]:
duckdb.__version__

'1.5.1'